In [ ]:
import requests

SERVER_URL = "http://localhost:8000"
MODEL_NAME = "Qwen2.5-VL-3B-Instruct"  # must match --served-model-name above

try:
    r = requests.get(f"{SERVER_URL}/v1/models", timeout=5)
    models = r.json()
    print("✅ Server is running!")
    print("Available models:", [m['id'] for m in models.get('data', [])])
except Exception as e:
    print("❌ Server is NOT running. Please start vLLM first.")
    print(f"   Error: {e}")

In [ ]:
from qwen_agent.agents import Assistant
from qwen_agent.utils.output_beautify import typewriter_print, multimodal_typewriter_print
print("✅ Imports successful")

In [ ]:
# NOTE: MODEL_NAME must exactly match --served-model-name used when starting vLLM
llm_cfg = {
    'model_type': 'qwenvl_oai',
    'model': MODEL_NAME,
    'model_server': f'{SERVER_URL}/v1',
    'api_key': 'EMPTY',
    'generate_cfg': {
        "top_p": 0.8,
        "top_k": 20,
        "temperature": 0.7,
        "repetition_penalty": 1.0,
        "presence_penalty": 1.5
    }
}
print(f"✅ LLM config ready — using model: {MODEL_NAME}")

In [ ]:
analysis_prompt = """Your role is that of a research assistant specializing in visual information. Answer questions about images by looking at them closely and then using research tools. Please follow this structured thinking process and show your work.

Start an iterative loop for each question:

- **First, look closely:** Begin with a detailed description of the image, paying attention to the user's question. List what you can tell just by looking, and what you'll need to look up.
- **Next, find information:** Use a tool to research the things you need to find out.
- **Then, review the findings:** Carefully analyze what the tool tells you and decide on your next action.

Continue this loop until your research is complete.

To finish, bring everything together in a clear, synthesized answer that fully responds to the user's question."""

tools = ['image_zoom_in_tool']
agent = Assistant(
    llm=llm_cfg,
    function_list=tools,
    system_message=analysis_prompt,
)
print("✅ Agent created")

In [ ]:
import os

IMAGE_PATH = "./resource/hopinn.jpg"  # <-- change this if your image is elsewhere

if not os.path.exists(IMAGE_PATH):
    print(f"⚠️  Warning: Image not found at '{IMAGE_PATH}'")
    print("   Please update IMAGE_PATH or place your image at that location.")
else:
    print(f"✅ Image found: {IMAGE_PATH}")

messages = []
messages += [
    {"role": "user", "content": [
        {"image": IMAGE_PATH},
        {"text": "Where was the picture taken?"}
    ]}
]

In [ ]:
print("Running agent...\n")
response_plain_text = ''
for ret_messages in agent.run(messages):
    # ret_messages contains interleaved assistant messages and tool responses
    response_plain_text = multimodal_typewriter_print(ret_messages, response_plain_text)
print("\n✅ Done")